In [1]:
import csv
import os
import pandas as pd
from cyvcf2 import VCF

In [2]:
def vcf_parsing(file_path: str) -> str:
    '''
    Parses the VCF file and extracts relevant data, then saves the processed data to a TSV file.

    Args:
    file_path (str): The path to the VCF file for parsing.

    Returns:
    str: A message indicating the creation of the TSV file.
    '''

    folder_path = 'processed_data'
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f'Folder "{folder_path}" created.')
    else:
        print(f'Folder "{folder_path}" exists.')

    # Extract the base name of the file (before .vcf or other extensions)
    basename = os.path.basename(file_path)
    name_without_extension = basename.split('.vcf')[0]

    destination_file = f'{folder_path}/{name_without_extension}_parsed.tsv'
    vcf = VCF(file_path)
    print('VCF file loaded.')
    
    # Define the columns to extract
    info_fields_to_extract = ['AC', 'AC_afr', 'AC_amr', 'AC_nfe',
                              'AC_asj', 'AC_sas', 'AC_eas', 'AC_mid', 'AC_fin',
                              'AN', 'AN_afr', 'AN_amr', 'AN_nfe', 'AN_asj',
                              'AN_sas', 'AN_eas', 'AN_mid', 'AN_fin',
                              'AF', 'AF_afr', 'AF_amr', 'AF_nfe', 'AF_asj',
                              'AF_sas', 'AF_eas', 'AF_mid', 'AF_fin', 'vep']

    vep_field_mapping = {
        1: 'Consequence', 2: 'IMPACT', 3: 'SYMBOL', 4: 'Gene',
        6: 'Feature', 7: 'BIOTYPE', 8: 'EXON', 9: 'INTRON', 
        12: 'cDNA_position', 13: 'CDS_position', 14: 'Protein_position', 15: 'Amino_acids', 16: 'Codons',
        17: 'ALLELE_NUM', 19: 'STRAND', 20: 'FLAGS', 21: 'VARIANT_CLASS',
        24: 'CANONICAL', 42: 'LoF', 43: 'LoF_filter',
        44: 'LoF_flags', 45: 'LoF_info'
    }

    column_names = ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'AC', 'AC_afr',
                    'AC_amr', 'AC_nfe', 'AC_asj', 'AC_sas', 'AC_eas',
                    'AC_mid', 'AC_fin', 'AN', 'AN_afr', 'AN_amr', 'AN_nfe',
                    'AN_asj', 'AN_sas', 'AN_eas', 'AN_mid', 'AN_fin', 'AF',
                    'AF_afr', 'AF_amr', 'AF_nfe', 'AF_asj', 'AF_sas', 'AF_eas',
                    'AF_mid', 'AF_fin', 'Consequence', 'IMPACT', 'SYMBOL', 'Gene',
                    'Feature', 'BIOTYPE', 'EXON', 'INTRON', 'cDNA_position', 
                    'CDS_position', 'Protein_position', 'Amino_acids', 'Codons',
                    'ALLELE_NUM', 'STRAND', 'FLAGS', 'VARIANT_CLASS', 'CANONICAL', 
                    'LoF', 'LoF_filter', 'LoF_flags', 'LoF_info']

    print('VCF parsing and streaming to file in progress...')

    with open(destination_file, 'w', newline='') as tsvfile:
        writer = csv.writer(tsvfile, delimiter='\t')
        writer.writerow(column_names)  # Write header
        
        # Iterate over each variant in the VCF file
        for variant in vcf:
            if 'PASS' in variant.FILTERS:
                variant_data = [variant.CHROM, variant.POS,
                                variant.ID, variant.REF, variant.ALT[0]]
                info_data = [variant.INFO.get(field, '.') for field in info_fields_to_extract]
                vep_annotation = variant.INFO.get('vep')

                # Handle multiple transcripts in vep if present
                if vep_annotation:
                    for transcript in vep_annotation.split(','):
                        split_transcript = transcript.split('|')
                        vep_fields = []
                        for key in vep_field_mapping.keys():
                            try:
                                vep_fields.append(split_transcript[key])
                            except Exception:
                                vep_fields.append('.')

                        # Conditions for row selection
                        vep_dict = dict(zip(vep_field_mapping.values(), vep_fields))

                        if (vep_dict.get('CANONICAL', '.') == 'YES' and
                            vep_dict.get('VARIANT_CLASS', '.') == 'SNV' and
                            vep_dict.get('Feature', '.').startswith('ENST')):

                            writer.writerow(variant_data + info_data[:-1] + vep_fields)

    print('File writing complete.')
    return f'{destination_file} file created'

In [ ]:
vcf_parsing("/home/bar-1/gnomad_v4_raw_data/gnomad.exomes.v4.1.sites.chr4.vcf.bgz")
with open("processed_data/chr4.txt", "w", encoding="utf-8") as f:
    f.write("chr4")

Folder "processed_data" exists.
VCF file loaded.
VCF parsing and streaming to file in progress...


In [ ]:
vcf_parsing("/home/bar-1/gnomad_v4_raw_data/gnomad.exomes.v4.1.sites.chr2.vcf.bgz")
with open("processed_data/chr2.txt", "w", encoding="utf-8") as f:
    f.write("chr2")

In [ ]:
vcf_parsing("/home/bar-1/gnomad_v4_raw_data/gnomad.exomes.v4.1.sites.chr19.vcf.bgz")
with open("processed_data/chr19.txt", "w", encoding="utf-8") as f:
    f.write("chr19")

In [ ]:
vcf_parsing("/home/bar-1/gnomad_v4_raw_data/gnomad.exomes.v4.1.sites.chr17.vcf.bgz")
with open("processed_data/chr17.txt", "w", encoding="utf-8") as f:
    f.write("chr17")

In [ ]:
vcf_parsing("/home/bar-1/gnomad_v4_raw_data/gnomad.exomes.v4.1.sites.chr15.vcf.bgz")
with open("processed_data/chr15.txt", "w", encoding="utf-8") as f:
    f.write("chr15")

In [ ]:
vcf_parsing("/home/bar-1/gnomad_v4_raw_data/gnomad.exomes.v4.1.sites.chr13.vcf.bgz")
with open("processed_data/chr13.txt", "w", encoding="utf-8") as f:
    f.write("chr13")

In [ ]:
vcf_parsing("/home/bar-1/gnomad_v4_raw_data/gnomad.exomes.v4.1.sites.chr11.vcf.bgz")
with open("processed_data/chr11.txt", "w", encoding="utf-8") as f:
    f.write("chr11")

In [ ]:
vcf_parsing("/home/bar-1/gnomad_v4_raw_data/gnomad.exomes.v4.1.sites.chr9.vcf.bgz")
with open("processed_data/chr9.txt", "w", encoding="utf-8") as f:
    f.write("chr9")

In [ ]:
vcf_parsing("/home/bar-1/gnomad_v4_raw_data/gnomad.exomes.v4.1.sites.chr7.vcf.bgz")
with open("processed_data/chr7.txt", "w", encoding="utf-8") as f:
    f.write("chr7")

In [ ]:
vcf_parsing("/home/bar-1/gnomad_v4_raw_data/gnomad.exomes.v4.1.sites.chr5.vcf.bgz")
with open("processed_data/chr5.txt", "w", encoding="utf-8") as f:
    f.write("chr5")

In [ ]:
vcf_parsing("/home/bar-1/gnomad_v4_raw_data/gnomad.exomes.v4.1.sites.chr3.vcf.bgz")
with open("processed_data/chr3.txt", "w", encoding="utf-8") as f:
    f.write("chr3")

In [ ]:
vcf_parsing("/home/bar-1/gnomad_v4_raw_data/gnomad.exomes.v4.1.sites.chr1.vcf.bgz")
with open("processed_data/chr1.txt", "w", encoding="utf-8") as f:
    f.write("chr1")

In [20]:
df = pd.read_csv("processed_data/gnomad.exomes.v4.1.sites.chr18_parsed.tsv", sep='\t')

/tmp/ipykernel_2982179/2093508694.py:1: DtypeWarning: Columns (26,31,51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("processed_data/gnomad.exomes.v4.1.sites.chr18_parsed.tsv", sep='\t')


In [21]:
df.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'AC', 'AC_afr', 'AC_amr', 'AC_nfe',
       'AC_asj', 'AC_sas', 'AC_eas', 'AC_mid', 'AC_fin', 'AN', 'AN_afr',
       'AN_amr', 'AN_nfe', 'AN_asj', 'AN_sas', 'AN_eas', 'AN_mid', 'AN_fin',
       'AF', 'AF_afr', 'AF_amr', 'AF_nfe', 'AF_asj', 'AF_sas', 'AF_eas',
       'AF_mid', 'AF_fin', 'Consequence', 'IMPACT', 'SYMBOL', 'Gene',
       'Feature', 'BIOTYPE', 'EXON', 'INTRON', 'cDNA_position', 'CDS_position',
       'Protein_position', 'Amino_acids', 'Codons', 'ALLELE_NUM', 'STRAND',
       'FLAGS', 'VARIANT_CLASS', 'CANONICAL', 'LoF', 'LoF_filter', 'LoF_flags',
       'LoF_info'],
      dtype='object')

In [8]:
# print(df.iloc[0])

In [22]:
df['FLAGS'].value_counts()

Series([], Name: count, dtype: int64)

In [23]:
df['LoF_flags'].value_counts()

LoF_flags
PHYLOCSF_WEAK            791
SINGLE_EXON              171
NAGNAG_SITE              115
NON_CAN_SPLICE            35
PHYLOCSF_UNLIKELY_ORF      6
.                          2
Name: count, dtype: int64

In [24]:
df['LoF_filter'].value_counts()

LoF_filter
END_TRUNC         464
5UTR_SPLICE       110
3UTR_SPLICE        16
GC_TO_GT_DONOR      7
ANC_ALLELE          1
Name: count, dtype: int64

In [25]:
df['LoF'].value_counts()

LoF
HC    10132
LC      598
Name: count, dtype: int64

In [26]:
df['LoF_info'].value_counts()

LoF_info
.                               63
INTRON_SIZE:93                  15
INTRON_SIZE:92                  13
INTRON_SIZE:1884                10
INTRON_SIZE:83                   9
                                ..
PERCENTILE:0.841831633673265     1
PERCENTILE:0.83003399320136      1
INTRON_SIZE:384                  1
PERCENTILE:0.829034193161368     1
PERCENTILE:0.942611477704459     1
Name: count, Length: 8171, dtype: int64

In [27]:
df['STRAND'].value_counts()

STRAND
 1    727799
-1    685562
Name: count, dtype: int64

In [28]:
df['CANONICAL'].value_counts()

CANONICAL
YES    1413361
Name: count, dtype: int64